# CVST / SII 完整可重复分析

本 Notebook 调用同目录下的 `cvst_complete_analysis.py`。该 Python 文件包含完整分析源码，而不是预先计算结果。

运行后自动生成：正文 Fig1–Fig6、补充 eFig01–eFig60、正文 Table1–Table5、补充 TableS01–TableS20。


In [ ]:
from pathlib import Path
import os, sys

# 请确保 Notebook、cvst_complete_analysis.py、data.csv 位于同一文件夹
PROJECT = Path.cwd()
print("Project:", PROJECT)
print("data.csv exists:", (PROJECT / "data.csv").exists())
print("analysis script exists:", (PROJECT / "cvst_complete_analysis.py").exists())


## 依赖检查

如缺包，请先在终端运行 `pip install -r requirements.txt`。正式代码不要求 shap/lime 第三方包。


In [ ]:
import numpy, pandas, scipy, sklearn, statsmodels, matplotlib, patsy, openpyxl
print("Core dependencies imported successfully.")


## 正式分析参数

`FAST_MODE=False` 为正式设置。若只想测试环境，可在 Python 源码中临时改为 True。


In [ ]:
import importlib
import cvst_complete_analysis as cvst
importlib.reload(cvst)
print("FAST_MODE =", cvst.FAST_MODE)
print("Bootstrap N =", cvst.BOOTSTRAP_N)
print("Repeated CV =", cvst.REPEATS, "x", cvst.FOLDS)
print("LASSO C grid =", len(cvst.C_GRID))


## 一键运行全部分析

该单元格是唯一需要长时间运行的部分。所有文件会自动保存，不依赖交互式显示。


In [ ]:
A = cvst.run_all()


## 完整性检查


In [ ]:
import pandas as pd
manifest = pd.read_csv(cvst.META_DIR / "output_manifest.csv")
display(manifest)

fig_ok = manifest.loc[manifest.Type.eq("Figure"), ["PDF exists", "PNG exists"]].fillna(False).all(axis=1).all()
tab_ok = manifest.loc[manifest.Type.eq("Table"), ["CSV exists", "XLSX exists"]].fillna(False).all(axis=1).all()
print("All 66 figures generated:", fig_ok)
print("All 25 tables generated:", tab_ok)


## 关键分析对象

`A` 中保存了原始数据、单因素结果、LASSO路径与共识变量、最终三变量模型、重复CV预测、完整拟合模型及模型归因。


In [ ]:
print("N =", len(A.df), "events =", int(A.df[cvst.Y].sum()))
print("Pre-screen variables:", [cvst.PRETTY.get(x,x) for x in A.prescreen])
print("Consensus variables:", [cvst.PRETTY.get(x,x) for x in A.consensus8])
display(A.final_or)
